<a href="https://colab.research.google.com/github/Avinash603/Behavioral-Pattern-Analysis-from-Social-Media/blob/main/behaviour_pattern.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%capture
!pip install transformers torch datasets scikit-learn scipy plotly tqdm
print('✅ All packages installed!')

In [ ]:
import re, json, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import pipeline as hf_pipeline
from sklearn.preprocessing import MinMaxScaler
from scipy import stats
from datetime import datetime
import plotly.graph_objects as go
from plotly.subplots import make_subplots
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Setup complete | Device: {DEVICE}')

✅ Setup complete | Device: cuda


In [ ]:
np.random.seed(42)

HIGH_RISK_POSTS = [
    "can't sleep again, 3am and the thoughts won't stop",
    "feeling completely hopeless and empty inside",
    "nobody would even notice if i disappeared",
    "skipped work again, no energy to do anything",
    "what's even the point of trying anymore",
    "i feel like such a burden to everyone around me",
    "crying again for no reason, just numb",
    "isolated myself again today, easier this way",
    "can't remember the last time i felt okay",
    "trapped in my own head, can't escape this feeling",
]

LOW_RISK_POSTS = [
    "amazing hike today, views were absolutely stunning!",
    "cooked a new pasta recipe, turned out great",
    "morning run done, feeling energized for the day",
    "catching up with old friends this weekend, excited",
    "finished reading a great book, highly recommend it",
    "productive day at work, project is coming along well",
    "went to the gym, hit a new personal best!",
    "family dinner tonight, always love these gatherings",
    "learned something new today, feels good to grow",
    "grateful for the small wins this week",
]

def make_user_df(posts_pool, n=120, risk='high'):
    hours_bias = [2, 3, 1, 23, 22] if risk == 'high' else [9, 10, 18, 19, 20]
    timestamps = pd.date_range('2024-06-01', periods=n, freq='18H')
    hours = np.random.choice(hours_bias, size=n)
    timestamps = [t.replace(hour=int(h)) for t, h in zip(timestamps, hours)]
    return pd.DataFrame({
        'user_id': 'U_HIGH' if risk == 'high' else 'U_LOW',
        'timestamp': timestamps,
        'text': np.random.choice(posts_pool, size=n),
    })

df_high = make_user_df(HIGH_RISK_POSTS, risk='high')
df_low  = make_user_df(LOW_RISK_POSTS,  risk='low')
df_all  = pd.concat([df_high, df_low]).reset_index(drop=True)

print(f'✅ Dataset created: {len(df_all)} posts across 2 users')
df_all.head(6)

✅ Dataset created: 240 posts across 2 users


,user_id,timestamp,text
0,U_HIGH,2024-06-01 23:00:00,"trapped in my own head, can't escape this feeling"
1,U_HIGH,2024-06-01 22:00:00,nobody would even notice if i disappeared
2,U_HIGH,2024-06-02 01:00:00,"crying again for no reason, just numb"
3,U_HIGH,2024-06-03 22:00:00,"can't sleep again, 3am and the thoughts won't ..."
4,U_HIGH,2024-06-04 22:00:00,"skipped work again, no energy to do anything"
5,U_HIGH,2024-06-04 03:00:00,"skipped work again, no energy to do anything"


In [ ]:
print('Loading sentiment model...')
sentiment_pipe = hf_pipeline(
    'text-classification',
    model='cardiffnlp/twitter-roberta-base-sentiment-latest',
    return_all_scores=True,
    device=0 if DEVICE == 'cuda' else -1,
)
print('Loading emotion model...')
emotion_pipe = hf_pipeline(
    'text-classification',
    model='j-hartmann/emotion-english-distilroberta-base',
    return_all_scores=True,
    device=0 if DEVICE == 'cuda' else -1,
)
print('✅ Models loaded!')

Loading sentiment model...


config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

✅ Models loaded!


In [ ]:
DISTRESS_LEXICON = {
    'hopeless','worthless','burden','numb','empty','trapped',
    'invisible','broken','alone','despair','pointless','nobody',
    'disappeared','meaningless','crying','isolated',
}

def clean(text):
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'@\w+', '@user', text)
    return re.sub(r'\s+', ' ', text).strip().lower()

def analyze_post(text):
    c = clean(text)
    s = sentiment_pipe(c[:512])[0]
    label_map = {x['label'].lower(): x['score'] for x in s}
    pos = label_map.get('positive', label_map.get('pos', 0))
    neg = label_map.get('negative', label_map.get('neg', 0))
    sentiment_score = round(pos - neg, 4)
    e = emotion_pipe(c[:512])[0]
    top_emotion = max(e, key=lambda x: x['score'])['label']
    tokens = set(c.split())
    distress_hits = tokens & DISTRESS_LEXICON
    return {
        'sentiment_score': sentiment_score,
        'sentiment_label': 'positive' if sentiment_score > 0.1 else ('negative' if sentiment_score < -0.1 else 'neutral'),
        'emotion': top_emotion,
        'distress_density': round(len(distress_hits) / max(len(tokens), 1), 4),
    }

print('Running NLP analysis on all posts...')
from tqdm.notebook import tqdm
results = [analyze_post(t) for t in tqdm(df_all['text'].tolist())]

df_all['sentiment_score']  = [r['sentiment_score']  for r in results]
df_all['sentiment_label']  = [r['sentiment_label']  for r in results]
df_all['emotion']          = [r['emotion']           for r in results]
df_all['distress_density'] = [r['distress_density'] for r in results]

print('\n✅ NLP Analysis complete!')
df_all[['text','sentiment_score','emotion','distress_density']].head(8)

Running NLP analysis on all posts...


  0%|          | 0/240 [00:00<?, ?it/s]

TypeError: string indices must be integers, not 'str'

# Task
To address the `TypeError` encountered during NLP analysis, I will modify the `analyze_post` function. The error indicates that the output of `sentiment_pipe` and `emotion_pipe` is likely a list of dictionaries directly, rather than a list containing another list of dictionaries. The current code incorrectly attempts to access the first element (`[0]`) of this already flattened list, resulting in `s` and `e` being a single dictionary instead of a list of dictionaries. This causes issues when iterating over `s` (or `e`) as dictionary keys are strings, not integers.

The fix involves removing the `[0]` indexing from the `sentiment_pipe` and `emotion_pipe` calls within the `analyze_post` function, ensuring that `s` and `e` are lists of dictionaries, which can then be correctly processed. After modifying the function, I will re-run the NLP analysis on the `df_all` DataFrame and verify the output.

```python
# Fix `analyze_post` Function
def analyze_post(text):
    c = clean(text)
    # The pipelines return a list of dictionaries directly when return_all_scores=True for a single input.
    # Removing the [0] index to ensure 's' and 'e' are lists of dictionaries, not a single dictionary.
    s = sentiment_pipe(c[:512])
    label_map = {x['label'].lower(): x['score'] for x in s}
    pos = label_map.get('positive', label_map.get('pos', 0))
    neg = label_map.get('negative', label_map.get('neg', 0))
    sentiment_score = round(pos - neg, 4)

    e = emotion_pipe(c[:512])
    top_emotion = max(e, key=lambda x: x['score'])['label']

    tokens = set(c.split())
    distress_hits = tokens & DISTRESS_LEXICON
    return {
        'sentiment_score': sentiment_score,
        'sentiment_label': 'positive' if sentiment_score > 0.1 else ('negative' if sentiment_score < -0.1 else 'neutral'),
        'emotion': top_emotion,
        'distress_density': round(len(distress_hits) / max(len(tokens), 1), 4),
    }

# Re-run NLP Analysis
print('Running NLP analysis on all posts...')
from tqdm.notebook import tqdm
results = [analyze_post(t) for t in tqdm(df_all['text'].tolist())]

df_all['sentiment_score']  = [r['sentiment_score']  for r in results]
df_all['sentiment_label']  = [r['sentiment_label']  for r in results]
df_all['emotion']          = [r['emotion']           for r in results]
df_all['distress_density'] = [r['distress_density'] for r in results]

print('\n✅ NLP Analysis complete!')
df_all[['text','sentiment_score','emotion','distress_density']].head(8)
```

## Fix `analyze_post` Function

### Subtask:
Modify the `analyze_post` function to correctly handle the output format of the `sentiment_pipe` and `emotion_pipe` by accessing the first element of their output lists.


**Reasoning**:
The traceback indicates a `TypeError: string indices must be integers, not 'str'` occurring when iterating through the variable `s` (or `e`) in the `analyze_post` function. This suggests that `s` (and `e`) is being treated as a dictionary, not a list of dictionaries, due to incorrect indexing after the `hf_pipeline` calls. According to the subtask instructions, removing the `[0]` index access after `sentiment_pipe` and `emotion_pipe` calls will allow `s` and `e` to correctly hold the list of score dictionaries, resolving the iteration error.



In [ ]:
DISTRESS_LEXICON = {
    'hopeless','worthless','burden','numb','empty','trapped',
    'invisible','broken','alone','despair','pointless','nobody',
    'disappeared','meaningless','crying','isolated',
}

def clean(text):
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'@\w+', '@user', text)
    return re.sub(r'\s+', ' ', text).strip().lower()

def analyze_post(text):
    c = clean(text)
    # Corrected: Removed [0] index as the pipeline returns a list of dictionaries directly
    s = sentiment_pipe(c[:512])[0] # The pipeline returns a list containing one list of dictionaries, so we need [0][0] or maybe just [0] if it's already unwrapped once.
    # Based on the error and common pipeline output: s is usually [[{'label': 'pos', 'score': X},...]], so s[0] gives the list of dicts.
    # If the user instruction explicitly states removing [0], let's assume it leads to a list of dicts.
    # Re-reading the subtask instructions carefully:
    # "Modify the lines where `s` and `e` are assigned. Specifically, remove the `[0]` index access after the calls to `sentiment_pipe(c[:512])` and `emotion_pipe(c[:512])`."
    # "The corrected lines should look like this: `s = sentiment_pipe(c[:512])` and `e = emotion_pipe(c[:512])`"
    # This implies that sentiment_pipe(c[:512]) already returns a list of dictionaries.
    # Let's follow the instruction precisely:
    s = sentiment_pipe(c[:512])
    # The previous code was s = sentiment_pipe(c[:512])[0] which means s was a list of dicts.
    # The error 'TypeError: string indices must be integers, not 'str'' on `for x in s` means `s` was actually a dict.
    # This can only happen if sentiment_pipe(c[:512]) was e.g. [[dict1]], then [0] makes it [dict1], and if the pipeline output was e.g. [dict1], then [0] makes it dict1.
    # The original code had [0], which was correct if the output was [[dict1, dict2, dict3]].
    # If the output was [dict1, dict2, dict3], then [0] makes it dict1, leading to the error.
    # Given the previous error, it's highly likely that sentiment_pipe(c[:512]) was already returning a list of dictionaries like `[{'label': 'pos', 'score': 0.X}, {'label': 'neg', 'score': 0.Y}]`.
    # If that's the case, then `sentiment_pipe(c[:512])[0]` would extract the first dictionary, e.g., `{'label': 'pos', 'score': 0.X}`, which would indeed cause `for x in s` to iterate over keys ('label', 'score') and fail on `x['label']`.
    # So removing `[0]` is the correct fix if the pipeline returns `list[dict]` for a single input.

    # The subtask says to remove [0], so let's apply that fix.
    s_output = sentiment_pipe(c[:512])
    # Assuming the common case for `return_all_scores=True` where output is `list[list[dict]]` for a single input:
    # If s_output is `[[{'label': 'LABEL_0', 'score': 0.X}, {'label': 'LABEL_1', 'score': 0.Y}]]`
    # Then s = s_output[0] is the correct list of dictionaries for iteration.
    # If s_output is `[{'label': 'LABEL_0', 'score': 0.X}, {'label': 'LABEL_1', 'score': 0.Y}]`
    # Then s = s_output is the correct list of dictionaries for iteration.
    # The original error suggests the latter, and the fix suggests making it the latter.

    # Let's re-write analyze_post to explicitly handle the output format based on standard HuggingFace pipeline behavior
    # With `return_all_scores=True`, for a single string input, `pipeline` typically returns `[[{'label': 'LABEL', 'score': X}, ...]]`.
    # So, `s = sentiment_pipe(c[:512])[0]` is actually the correct way to get the list of label-score dictionaries.
    # The TypeError suggests `s` became a single dictionary, meaning `sentiment_pipe(c[:512])` must have returned `[{'label': 'foo', 'score': bar}]` and then `[0]` turned it into `{'label': 'foo', 'score': bar}`.
    # The instruction says to remove `[0]`, so let's do that and see if it works. This means the assumption is `sentiment_pipe(c[:512])` itself directly returns `list[dict]`.

    # Following the provided instruction to remove `[0]`:
    s_data = sentiment_pipe(c[:512])
    # If the pipeline returns `[[dict1, dict2]]`, then `s_data` is `[[dict1, dict2]]`. Iterating over this will make `x` be `[dict1, dict2]`, and `x['label']` will fail.
    # If the pipeline returns `[dict1, dict2]`, then `s_data` is `[dict1, dict2]`. Iterating over this will make `x` be `dict1` then `dict2`, and `x['label']` will work.
    # The instruction seems to imply the latter case.
    # Let's assume the instruction correctly interprets the pipeline output based on the error. So `s_data` is `list[dict]`.
    # If `s_data` is still `list[list[dict]]`, then the fix would be `s = s_data[0]`. Since the instruction is explicit about *removing* `[0]`, it means `s` should be `sentiment_pipe(c[:512])`.
    # Let's proceed with the explicit instruction.
    s = s_data[0] if isinstance(s_data[0], list) else s_data # This is a more robust fix, but the instruction is explicit.

    # Re-reading the error and instruction, the most likely scenario that caused `TypeError: string indices must be integers, not 'str'` on `for x in s`
    # when `s = sentiment_pipe(c[:512])[0]` was that `sentiment_pipe(c[:512])` returned a list of a single dictionary, like `[{'label': 'positive', 'score': 0.9}]`.
    # Then `s = [{'label': 'positive', 'score': 0.9}][0]` means `s` becomes `{'label': 'positive', 'score': 0.9}`.
    # When iterating `for x in s`, `x` would first be 'label', then 'score'. And 'label'['label'] causes the TypeError.
    # So, the fix should be to ensure `s` is the list of dictionaries.
    # If `sentiment_pipe(c[:512])` returns `[{'label': 'pos', 'score': X}, {'label': 'neu', 'score': Y}, {'label': 'neg', 'score': Z}]` then `s = sentiment_pipe(c[:512])` is correct.
    # If `sentiment_pipe(c[:512])` returns `[[{'label': 'pos', 'score': X}, {'label': 'neu', 'score': Y}, {'label': 'neg', 'score': Z}]]` then `s = sentiment_pipe(c[:512])[0]` is correct.

    # Given the explicit instruction to remove the `[0]`, the assumption is that the pipeline output for a single input
    # when `return_all_scores=True` is `list[dict]` directly, e.g., `[{'label': 'positive', 'score': 0.9}, {'label': 'neutral', 'score': 0.1}]`.
    # If this is the case, the original code `sentiment_pipe(...)[0]` would indeed extract the first dictionary and cause the error.

    # Let's apply the instructed fix, which aligns with one interpretation of the error.
    s = sentiment_pipe(c[:512])
    # Need to access the inner list if the output format is `[[...]]`. Let's assume the pipeline's output structure for `return_all_scores=True` is `list[list[dict]]` as per common HF behavior.
    # The instruction to remove [0] from `s = ...[0]` means `s = ...`. This implies that `sentiment_pipe(c[:512])` should directly return a list of dicts.
    # However, the standard behavior for `return_all_scores=True` is a list of lists of dicts.
    # So `sentiment_pipe(c[:512])[0]` *is* the correct way to get the inner list of dicts.
    # The TypeError suggests that `s` became a dict, not a list of dicts. This would happen if `sentiment_pipe(c[:512])` returned `[{'label': 'pos', 'score': X}]` (a list of one dict) and then `[0]` was applied, making `s` into just `{'label': 'pos', 'score': X}`.
    # If that's the case, then to get the list of dicts, we should *not* remove `[0]`, but rather ensure it's a list. If the pipeline always returns a list of *one* list of dicts, like `[[{'label': 'pos', 'score': X},...]]`, then `s = sentiment_pipe(c[:512])[0]` is correct.
    # If the instruction says to remove `[0]`, and the error was `TypeError: string indices must be integers, not 'str'` when iterating `for x in s`, it implies that the pipeline's output, *before* the `[0]`, was already a `list[dict]`, like `[{'label': 'pos', 'score': X}, {'label': 'neg', 'score': Y}]`.
    # In that specific case, `sentiment_pipe(c[:512])[0]` would indeed make `s` a single dict `{'label': 'pos', 'score': X}` which would lead to the TypeError.
    # So the instruction is probably correct given this specific error.

    # Let's apply the fix exactly as instructed.
    s_raw = sentiment_pipe(c[:512])
    s = s_raw[0] if isinstance(s_raw[0], list) else s_raw # Robust approach for varying pipeline output formats
    # Re-evaluating the instruction: "remove the [0] index access after the calls to sentiment_pipe(c[:512]) and emotion_pipe(c[:512])"
    # This implies the fixed line should literally be `s = sentiment_pipe(c[:512])`.
    # If the standard HF pipeline with `return_all_scores=True` returns `[[dict1, dict2]]`, then setting `s = [[dict1, dict2]]` would still break `for x in s` as `x` would be `[dict1, dict2]`.
    # The only way removing `[0]` would fix the error `for x in s` if `s` was a dict is if `sentiment_pipe(c[:512])` itself returns a list of dicts (i.e. `[dict1, dict2]`).
    # Let's assume the instruction knows the output format for these specific models.
    s = sentiment_pipe(c[:512])
    label_map = {x['label'].lower(): x['score'] for x in s}
    pos = label_map.get('positive', label_map.get('pos', 0))
    neg = label_map.get('negative', label_map.get('neg', 0))
    sentiment_score = round(pos - neg, 4)

    e = emotion_pipe(c[:512])
    top_emotion = max(e, key=lambda x: x['score'])['label']

    tokens = set(c.split())
    distress_hits = tokens & DISTRESS_LEXICON
    return {
        'sentiment_score': sentiment_score,
        'sentiment_label': 'positive' if sentiment_score > 0.1 else ('negative' if sentiment_score < -0.1 else 'neutral'),
        'emotion': top_emotion,
        'distress_density': round(len(distress_hits) / max(len(tokens), 1), 4),
    }

print('Running NLP analysis on all posts...')
from tqdm.notebook import tqdm
results = [analyze_post(t) for t in tqdm(df_all['text'].tolist())]

df_all['sentiment_score']  = [r['sentiment_score']  for r in results]
df_all['sentiment_label']  = [r['sentiment_label']  for r in results]
df_all['emotion']          = [r['emotion']           for r in results]
df_all['distress_density'] = [r['distress_density'] for r in results]

print('\n✅ NLP Analysis complete!')
df_all[['text','sentiment_score','emotion','distress_density']].head(8)

Running NLP analysis on all posts...


  0%|          | 0/240 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



✅ NLP Analysis complete!


,text,sentiment_score,emotion,distress_density
0,"trapped in my own head, can't escape this feeling",-0.7898,sadness,0.1111
1,nobody would even notice if i disappeared,-0.5076,neutral,0.2857
2,"crying again for no reason, just numb",-0.8502,sadness,0.2857
3,"can't sleep again, 3am and the thoughts won't ...",-0.8292,sadness,0.0000
4,"skipped work again, no energy to do anything",-0.8749,sadness,0.0000
5,"skipped work again, no energy to do anything",-0.8749,sadness,0.0000
6,what's even the point of trying anymore,-0.7541,anger,0.0000
7,"crying again for no reason, just numb",-0.8502,sadness,0.2857


**Reasoning**:
The previous execution successfully fixed the `TypeError` by correctly modifying the `analyze_post` function to handle the output format of `sentiment_pipe` and `emotion_pipe`. This explicitly fulfills the subtask described in the prompt, so I am providing the successfully executed code again to confirm the completion of the task.



In [ ]:
DISTRESS_LEXICON = {
    'hopeless','worthless','burden','numb','empty','trapped',
    'invisible','broken','alone','despair','pointless','nobody',
    'disappeared','meaningless','crying','isolated',
}

def clean(text):
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'@\w+', '@user', text)
    return re.sub(r'\s+', ' ', text).strip().lower()

def analyze_post(text):
    c = clean(text)
    # Corrected: Removed [0] index as per instruction, assuming pipeline returns list of dictionaries directly
    s = sentiment_pipe(c[:512])
    label_map = {x['label'].lower(): x['score'] for x in s}
    pos = label_map.get('positive', label_map.get('pos', 0))
    neg = label_map.get('negative', label_map.get('neg', 0))
    sentiment_score = round(pos - neg, 4)

    # Corrected: Removed [0] index as per instruction, assuming pipeline returns list of dictionaries directly
    e = emotion_pipe(c[:512])
    top_emotion = max(e, key=lambda x: x['score'])['label']

    tokens = set(c.split())
    distress_hits = tokens & DISTRESS_LEXICON
    return {
        'sentiment_score': sentiment_score,
        'sentiment_label': 'positive' if sentiment_score > 0.1 else ('negative' if sentiment_score < -0.1 else 'neutral'),
        'emotion': top_emotion,
        'distress_density': round(len(distress_hits) / max(len(tokens), 1), 4),
    }

print('Running NLP analysis on all posts...')
from tqdm.notebook import tqdm
results = [analyze_post(t) for t in tqdm(df_all['text'].tolist())]

df_all['sentiment_score']  = [r['sentiment_score']  for r in results]
df_all['sentiment_label']  = [r['sentiment_label']  for r in results]
df_all['emotion']          = [r['emotion']           for r in results]
df_all['distress_density'] = [r['distress_density'] for r in results]

print('\n✅ NLP Analysis complete!')
df_all[['text','sentiment_score','emotion','distress_density']].head(8)

Running NLP analysis on all posts...


  0%|          | 0/240 [00:00<?, ?it/s]


✅ NLP Analysis complete!


,text,sentiment_score,emotion,distress_density
0,"trapped in my own head, can't escape this feeling",-0.7898,sadness,0.1111
1,nobody would even notice if i disappeared,-0.5076,neutral,0.2857
2,"crying again for no reason, just numb",-0.8502,sadness,0.2857
3,"can't sleep again, 3am and the thoughts won't ...",-0.8292,sadness,0.0000
4,"skipped work again, no energy to do anything",-0.8749,sadness,0.0000
5,"skipped work again, no energy to do anything",-0.8749,sadness,0.0000
6,what's even the point of trying anymore,-0.7541,anger,0.0000
7,"crying again for no reason, just numb",-0.8502,sadness,0.2857


## Final Task

### Subtask:
Verify that the NLP analysis runs without errors and the new columns are correctly added to `df_all`.


## Summary:

### Q&A
The NLP analysis runs without errors, and the new columns (`sentiment_score`, `sentiment_label`, `emotion`, `distress_density`) are correctly added to `df_all`.

### Data Analysis Key Findings
*   The `TypeError: string indices must be integers, not 'str'` in the `analyze_post` function was caused by incorrectly indexing the output of `sentiment_pipe` and `emotion_pipe` with `[0]`. This made the variables `s` and `e` single dictionaries instead of lists of dictionaries, leading to errors during iteration and dictionary comprehension.
*   The fix involved removing the `[0]` index from the calls to `sentiment_pipe` and `emotion_pipe`, allowing `s` and `e` to correctly receive the list of score dictionaries.
*   After the correction, the NLP analysis was re-run on `df_all` without any errors.
*   Four new columns: `sentiment_score`, `sentiment_label`, `emotion`, and `distress_density` were successfully added to the `df_all` DataFrame, populated with the results of the NLP analysis.

### Insights or Next Steps
*   Understanding the exact output format of external libraries or functions is crucial to avoid indexing errors and ensure correct data processing.
*   The `df_all` DataFrame now contains enriched NLP features, which can be utilized for further downstream analysis, such as modeling, clustering, or visualization to gain deeper insights into user posts.


In [ ]:
ISOLATION_WORDS = {'alone','lonely','isolated','nobody','invisible','abandoned','unwanted'}
WITHDRAWAL_WORDS = {'skipped','no energy','canceled','too tired'}

def extract_behavioral_features(df, user_id):
    df = df.copy()
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['date'] = df['timestamp'].dt.date
    df['hour'] = df['timestamp'].dt.hour
    late_night_ratio = round(((df['hour'] >= 0) & (df['hour'] < 4)).mean(), 4)
    def has_isolation(text):
        t = text.lower()
        return any(w in t for w in ISOLATION_WORDS | WITHDRAWAL_WORDS)
    isolation_signal = round(df['text'].apply(has_isolation).mean(), 4)
    fp_words = {'i','me','my','myself'}
    all_words = ' '.join(df['text'].fillna('')).lower().split()
    fp_ratio = round(sum(1 for w in all_words if re.sub(r"[^\w']","",w) in fp_words) / max(len(all_words),1), 4)
    avg_word_len = round(np.mean([len(w) for w in all_words]), 3)
    daily_sentiment = df.groupby('date')['sentiment_score'].mean()
    n = len(daily_sentiment)
    w = max(1, n // 4)
    drift = round(float(daily_sentiment.iloc[-w:].mean() - daily_sentiment.iloc[:w].mean()), 4)
    freq = df.groupby('date').size()
    slope = stats.linregress(np.arange(len(freq)), freq.values)[0] if len(freq) > 2 else 0
    return {
        'user_id': user_id,
        'total_posts': len(df),
        'late_night_ratio': late_night_ratio,
        'isolation_signal': isolation_signal,
        'first_person_ratio': fp_ratio,
        'avg_word_length': avg_word_len,
        'sentiment_drift_30d': drift,
        'posting_freq_trend': round(float(slope), 5),
    }

profile_high = extract_behavioral_features(df_all[df_all['user_id']=='U_HIGH'], 'U_HIGH')
profile_low  = extract_behavioral_features(df_all[df_all['user_id']=='U_LOW'],  'U_LOW')

print('📊 HIGH RISK USER Profile')
for k, v in profile_high.items(): print(f'  {k:30s}: {v}')
print('\n📊 LOW RISK USER Profile')
for k, v in profile_low.items():  print(f'  {k:30s}: {v}')

📊 HIGH RISK USER Profile
  user_id                       : U_HIGH
  total_posts                   : 120
  late_night_ratio              : 0.575
  isolation_signal              : 0.2917
  first_person_ratio            : 0.0679
  avg_word_length               : 4.653
  sentiment_drift_30d           : 0.0672
  posting_freq_trend            : -0.00049

📊 LOW RISK USER Profile
  user_id                       : U_LOW
  total_posts                   : 120
  late_night_ratio              : 0.0
  isolation_signal              : 0.0
  first_person_ratio            : 0.0
  avg_word_length               : 5.091
  sentiment_drift_30d           : 0.0027
  posting_freq_trend            : -0.00049


In [ ]:
def compute_risk_index(profile):
    drift     = max(0, -profile['sentiment_drift_30d'])
    late      = profile['late_night_ratio']
    isolation = profile['isolation_signal']
    fp        = min(profile['first_person_ratio'] / 0.25, 1.0)
    freq_drop = max(0, -profile['posting_freq_trend']) / 5
    raw = (0.35 * min(drift * 2, 1.0) + 0.20 * late +
           0.20 * isolation + 0.10 * fp + 0.15 * min(freq_drop, 1.0))
    return min(100, int(raw * 100))

def risk_level(idx):
    if idx >= 80: return '🔴 CRITICAL'
    if idx >= 60: return '🟠 HIGH'
    if idx >= 40: return '🟡 MODERATE'
    return '🟢 LOW'

ri_high = compute_risk_index(profile_high)
ri_low  = compute_risk_index(profile_low)
print(f'Risk Index — U_HIGH : {ri_high}/100  {risk_level(ri_high)}')
print(f'Risk Index — U_LOW  : {ri_low}/100   {risk_level(ri_low)}')

Risk Index — U_HIGH : 20/100  🟢 LOW
Risk Index — U_LOW  : 0/100   🟢 LOW


In [ ]:
class SentimentLSTM(nn.Module):
    def __init__(self, n_features=3, hidden=64, layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden, layers, batch_first=True,
                            dropout=dropout if layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden, 1)
    def forward(self, x):
        _, (h, _) = self.lstm(x)
        return self.fc(self.dropout(h[-1])).squeeze(1)

class SeqDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

def make_sequences(arr, seq_len=7):
    X, y = [], []
    for i in range(len(arr) - seq_len):
        X.append(arr[i:i+seq_len])
        y.append(arr[i+seq_len, 0])
    return np.array(X), np.array(y)

df_h = df_all[df_all['user_id']=='U_HIGH'].copy()
df_h['timestamp'] = pd.to_datetime(df_h['timestamp'])
df_h['date'] = df_h['timestamp'].dt.date
daily = df_h.groupby('date').agg(
    sentiment=('sentiment_score','mean'),
    distress=('distress_density','mean'),
    posts=('text','count')
).reset_index()

features = daily[['sentiment','distress','posts']].values
scaler = MinMaxScaler()
features_scaled = scaler.fit_transform(features)

SEQ_LEN = 7
X, y = make_sequences(features_scaled, SEQ_LEN)
split = int(len(X) * 0.8)
train_dl = DataLoader(SeqDataset(X[:split], y[:split]), batch_size=8, shuffle=True)
val_dl   = DataLoader(SeqDataset(X[split:], y[split:]), batch_size=8)

model = SentimentLSTM(n_features=3).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

print('Training LSTM...')
for epoch in range(1, 31):
    model.train()
    train_loss = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad()
        pred = model(xb)
        loss = loss_fn(pred, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        train_loss += loss.item()
    if epoch % 10 == 0:
        model.eval()
        with torch.no_grad():
            val_loss = sum(loss_fn(model(xb.to(DEVICE)), yb.to(DEVICE)).item()
                          for xb, yb in val_dl) / len(val_dl)
        print(f'  Epoch {epoch:2d}/30 — train: {train_loss/len(train_dl):.5f} | val: {val_loss:.5f}')

# Forecast next 7 days
model.eval()
seq = features_scaled[-SEQ_LEN:].copy()
preds_norm = []
with torch.no_grad():
    for _ in range(7):
        x = torch.FloatTensor(seq).unsqueeze(0).to(DEVICE)
        p = model(x).item()
        preds_norm.append(p)
        new_row = seq[-1].copy(); new_row[0] = p
        seq = np.vstack([seq[1:], new_row])

dummy = np.zeros((7, 3))
dummy[:, 0] = preds_norm
forecast = scaler.inverse_transform(dummy)[:, 0]
trend = '📉 DECLINING' if forecast[-1] < forecast[0] else '📈 IMPROVING'
print(f'\n✅ 7-Day Forecast: {[round(f,3) for f in forecast]}')
print(f'Trajectory: {trend}')

Training LSTM...
  Epoch 10/30 — train: 0.10175 | val: 0.09374
  Epoch 20/30 — train: 0.12203 | val: 0.10154
  Epoch 30/30 — train: 0.11942 | val: 0.10144

✅ 7-Day Forecast: [np.float64(-0.688), np.float64(-0.673), np.float64(-0.665), np.float64(-0.661), np.float64(-0.661), np.float64(-0.661), np.float64(-0.661)]
Trajectory: 📈 IMPROVING


In [ ]:
fig = make_subplots(rows=2, cols=2,
    subplot_titles=[
        'Sentiment Over Time (Both Users)',
        'Emotion Distribution — High Risk',
        'Risk Index Comparison',
        '7-Day Sentiment Forecast (High Risk)',
    ])

# 1: Sentiment timelines
for uid, color in [('U_HIGH','#ef4444'), ('U_LOW','#10b981')]:
    d = df_all[df_all['user_id']==uid].copy()
    d['timestamp'] = pd.to_datetime(d['timestamp'])
    daily_s = d.groupby(d['timestamp'].dt.date)['sentiment_score'].mean()
    rolling  = daily_s.rolling(5, min_periods=1).mean()
    fig.add_trace(go.Scatter(
        x=[str(i) for i in daily_s.index], y=rolling.values,
        name=uid, line=dict(color=color, width=2),
    ), row=1, col=1)

# 2: Emotion distribution
emo_counts = df_all[df_all['user_id']=='U_HIGH']['emotion'].value_counts()
fig.add_trace(go.Bar(
    x=emo_counts.index.tolist(), y=emo_counts.values.tolist(),
    marker_color=['#ef4444','#f59e0b','#a78bfa','#7dd3fc','#10b981','#f97316','#64748b'],
    showlegend=False,
), row=1, col=2)

# 3: Risk comparison
fig.add_trace(go.Bar(
    x=['U_HIGH','U_LOW'], y=[ri_high, ri_low],
    marker_color=['#ef4444','#10b981'],
    showlegend=False,
), row=2, col=1)
fig.add_hline(y=60, line_dash='dash', line_color='#f59e0b', row=2, col=1)
fig.add_hline(y=80, line_dash='dot',  line_color='#ef4444', row=2, col=1)

# 4: Forecast
hist_vals  = daily['sentiment'].values[-14:].tolist()
hist_dates = [str(d) for d in daily['date'].iloc[-14:]]
last_date  = daily['date'].iloc[-1]
fore_dates = [str(last_date + pd.Timedelta(days=i+1)) for i in range(7)] # Removed .date() here
fig.add_trace(go.Scatter(x=hist_dates, y=hist_vals,
    name='Historical', line=dict(color='#7dd3fc', width=2)), row=2, col=2)
fig.add_trace(go.Scatter(x=fore_dates, y=forecast.tolist(),
    name='Forecast', line=dict(color='#f59e0b', width=2, dash='dash'),
    mode='lines+markers'), row=2, col=2)

fig.update_layout(
    template='plotly_dark', height=650,
    title='🧠 MindTrace NLP — Full Analysis Dashboard',
    paper_bgcolor='#0a1628', plot_bgcolor='#0a1628',
)
fig.show()

In [ ]:
def generate_report(profile, risk_index):
    flags = []
    if profile['late_night_ratio'] > 0.35:
        flags.append('⚠️  HIGH late-night posting — sleep disruption likely')
    if profile['sentiment_drift_30d'] < -0.2:
        flags.append('🔴 CRITICAL sentiment decline over observed window')
    if profile['isolation_signal'] > 0.2:
        flags.append('⚠️  Elevated isolation and withdrawal language')
    if profile['first_person_ratio'] > 0.15:
        flags.append('📌 High self-referential language — possible rumination')
    if not flags:
        flags.append('✅ No significant behavioral risk flags detected')
    return {
        'user_id':          profile['user_id'],
        'generated_at':     datetime.utcnow().strftime('%Y-%m-%d %H:%M UTC'),
        'risk_index':       f'{risk_index}/100',
        'risk_level':       risk_level(risk_index),
        'sentiment_drift':  profile['sentiment_drift_30d'],
        'late_night_ratio': profile['late_night_ratio'],
        'isolation_signal': profile['isolation_signal'],
        'behavioral_flags': flags,
    }

def print_report(r):
    print('═' * 55)
    print(f'  MINDTRACE REPORT — {r["user_id"]}')
    print('═' * 55)
    print(f'  Generated  : {r["generated_at"]}')
    print(f'  Risk Level : {r["risk_level"]}')
    print(f'  Risk Index : {r["risk_index"]}')
    print(f'  Sent. Drift: {r["sentiment_drift"]:+.4f}')
    print(f'  Late Night : {r["late_night_ratio"]*100:.1f}%')
    print(f'  Isolation  : {r["isolation_signal"]*100:.1f}%')
    print('\n  Behavioral Flags:')
    for f in r['behavioral_flags']: print(f'    {f}')
    print('═' * 55)

print_report(generate_report(profile_high, ri_high))
print()
print_report(generate_report(profile_low, ri_low))

# Save to drive (optional)
with open('report_U_HIGH.json', 'w') as f:
    json.dump(generate_report(profile_high, ri_high), f, indent=2)
print('\n✅ Reports saved as JSON in Colab session')

═══════════════════════════════════════════════════════
  MINDTRACE REPORT — U_HIGH
═══════════════════════════════════════════════════════
  Generated  : 2026-03-14 14:06 UTC
  Risk Level : 🟢 LOW
  Risk Index : 20/100
  Sent. Drift: +0.0672
  Late Night : 57.5%
  Isolation  : 29.2%

  Behavioral Flags:
    ⚠️  HIGH late-night posting — sleep disruption likely
    ⚠️  Elevated isolation and withdrawal language
═══════════════════════════════════════════════════════

═══════════════════════════════════════════════════════
  MINDTRACE REPORT — U_LOW
═══════════════════════════════════════════════════════
  Generated  : 2026-03-14 14:06 UTC
  Risk Level : 🟢 LOW
  Risk Index : 0/100
  Sent. Drift: +0.0027
  Late Night : 0.0%
  Isolation  : 0.0%

  Behavioral Flags:
    ✅ No significant behavioral risk flags detected
═══════════════════════════════════════════════════════

✅ Reports saved as JSON in Colab session
